# Projet 5 | Classification d'articles de presse (Partie II)

Notebook d'entraînement et de comparaison.

**CSV** : `data/dataset_valide.csv`

| Colonne | Description |
|---------|-------------|
| `titre` | Titre de l'article |
| `contenu_nettoye` | Texte nettoyé |
| `categorie` | Étiquette validée |



In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import Dataset
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

# Depuis notebooks/ → remonter à la racine du projet
NOTEBOOK_DIR = Path.cwd()
BASE_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_PATH = BASE_DIR / "data" / "dataset_valide.csv"
MODEL_DIR = BASE_DIR / "models" / "camembert-classifieur"
RANDOM_STATE = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("CSV:", DATA_PATH)

## 1. Chargement des données

In [ ]:
df = pd.read_csv(DATA_PATH)
df["texte"] = (df["titre"].astype(str) + " " + df["contenu_nettoye"].astype(str)).str.strip()
df = df[df["categorie"].notna() & (df["categorie"].astype(str).str.len() > 0)]
df = df[df["texte"].str.len() > 50].reset_index(drop=True)

print(f"Articles : {len(df)}")
print(f"Catégories : {df['categorie'].nunique()}")
df["categorie"].value_counts()

## 2. Split train / test

In [ ]:
labels = sorted(df["categorie"].unique())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

# Stratification possible uniquement si chaque classe a >= 2 articles
counts = df["categorie"].value_counts()
can_stratify = counts.min() >= 2
if not can_stratify:
    rares = counts[counts < 2].index.tolist()
    print("Attention : classes trop rares pour la stratification:", rares)

y_all = df["categorie"].map(label2id)
X_train, X_test, y_train, y_test = train_test_split(
    df["texte"],
    y_all,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_all if can_stratify else None,
)

print("Train:", len(X_train), "| Test:", len(X_test))
print("Labels:", labels)

## 3. Baselines classiques (BoW, TF-IDF)

Comparaison demandée par le sujet : vectorisation + classifieurs simples.

In [ ]:
def eval_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted")
    print(f"\n=== {name} ===")
    print(f"Accuracy: {acc:.4f} | F1 pondéré: {f1:.4f}")
    print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))
    return {"modele": name, "accuracy": acc, "f1_weighted": f1}


baselines = {
    "BoW + Naive Bayes": Pipeline([
        ("vec", CountVectorizer(max_features=20000, ngram_range=(1, 2))),
        ("clf", MultinomialNB()),
    ]),
    "TF-IDF + SVM": Pipeline([
        ("vec", TfidfVectorizer(max_features=20000, ngram_range=(1, 2))),
        ("clf", LinearSVC()),
    ]),
    "TF-IDF + Régression logistique": Pipeline([
        ("vec", TfidfVectorizer(max_features=20000, ngram_range=(1, 2))),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
}

results = []
for name, pipe in baselines.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    results.append(eval_model(name, y_test, preds))

## 3 bis. Word2Vec + classifieur

On entraîne un Word2Vec sur le corpus, puis chaque document est représenté par la **moyenne** des vecteurs de ses mots. Un classifieur (régression logistique) apprend ensuite sur ces embeddings.

In [ ]:
import re

from gensim.models import Word2Vec


def tokenize_simple(texte: str) -> list[str]:
    return re.findall(r"\b\w[\w'-]*\b", str(texte).lower())


train_tokens = [tokenize_simple(t) for t in X_train]
test_tokens = [tokenize_simple(t) for t in X_test]

W2V_DIM = 100
w2v = Word2Vec(
    sentences=train_tokens,
    vector_size=W2V_DIM,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    seed=RANDOM_STATE,
)


def document_vector(tokens: list[str]) -> np.ndarray:
    vecs = [w2v.wv[w] for w in tokens if w in w2v.wv]
    if not vecs:
        return np.zeros(W2V_DIM, dtype=np.float32)
    return np.mean(vecs, axis=0)


X_train_w2v = np.vstack([document_vector(t) for t in train_tokens])
X_test_w2v = np.vstack([document_vector(t) for t in test_tokens])

w2v_clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
w2v_clf.fit(X_train_w2v, y_train)
w2v_preds = w2v_clf.predict(X_test_w2v)

results.append(eval_model("Word2Vec + Régression logistique", y_test, w2v_preds))

## 4. CamemBERT | fine-tuning

Modèle pré-entraîné français : `camembert-base`

In [ ]:
MODEL_NAME = "camembert-base"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)


def tokenize_batch(batch):
    return tokenizer(batch["texte"], truncation=True, max_length=MAX_LENGTH)


train_ds = Dataset.from_dict({"texte": X_train.tolist(), "label": y_train.tolist()})
test_ds = Dataset.from_dict({"texte": X_test.tolist(), "label": y_test.tolist()})

train_ds = train_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels_eval = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels_eval, preds),
        "f1_weighted": f1_score(labels_eval, preds, average="weighted"),
    }


MODEL_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## 5. Évaluation CamemBERT

In [ ]:
camembert_preds = trainer.predict(test_ds)
y_pred_cam = np.argmax(camembert_preds.predictions, axis=-1)

results.append(eval_model("CamemBERT", y_test, y_pred_cam))

cm = confusion_matrix(y_test, y_pred_cam)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=labels, yticklabels=labels, cmap="Blues")
plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.title("Matrice de confusion | CamemBERT")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Comparaison finale & sauvegarde

In [ ]:
comparison = pd.DataFrame(results).sort_values("f1_weighted", ascending=False)
display(comparison)

plt.figure(figsize=(8, 4))
sns.barplot(data=comparison, x="f1_weighted", y="modele", palette="viridis")
plt.xlabel("F1 pondéré")
plt.title("Comparaison des modèles")
plt.tight_layout()
plt.show()

# Sauvegarde du modèle CamemBERT fine-tuné
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
comparison.to_csv(BASE_DIR / "data" / "resultats_comparaison.csv", index=False)
print("Modèle sauvegardé dans:", MODEL_DIR)

## 7. Test sur un nouvel article

In [ ]:
def predict_text(texte: str, top_k: int = 3):
    inputs = tokenizer(texte, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    inputs = {k: v.to(trainer.model.device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = trainer.model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    top_idx = probs.argsort()[::-1][:top_k]
    for i in top_idx:
        print(f"{id2label[i]}: {probs[i]:.2%}")


# Exemple
exemple = "Le gouvernement adopte de nouvelles mesures économiques pour soutenir les PME."
predict_text(exemple)